In [ ]:
import sys
import os
sys.path.append(os.path.abspath('Multi-Agent-Initialization'))

# Structured Web Data Extraction Agent - Version 5

This version introduces **Parallel Execution** (Map-Reduce style multi-source scraping). Instead of a single Web Researcher sequentially searching, the Planner now generates multiple distinct sub-queries or URLs, and the graph spawns parallel `web_researcher` nodes using LangGraph's native `Send` API to retrieve context simultaneously, greatly speeding up extraction from diverse sources.

In [ ]:
import os
import json
import warnings
import re
import requests
from dotenv import load_dotenv

warnings.filterwarnings('ignore')
load_dotenv()

import operator
from typing import Annotated, Literal, Optional, List, Dict, Any
from langchain_core.messages import HumanMessage
from langgraph.graph import MessagesState, START, StateGraph, END
from langgraph.types import Command, Send
from langgraph.prebuilt import create_react_agent
from langchain_groq import ChatGroq
from langchain_community.tools.tavily_search import TavilySearchResults

import prompts

# Setup LLMs
reasoning_llm = ChatGroq(model='llama-3.1-8b-instant')
llm = reasoning_llm

# Dedicated Sub-state for Parallel Tracking
class SearchTask:
    query: str
    agent: str

class State(MessagesState):
    user_query: str
    search_tasks: List[str] # List of sub-queries to execute in parallel
    scraped_contexts: Annotated[List[str], operator.add] # Aggregated results
    final_answer: str

In [ ]:
tavily_tool = TavilySearchResults(max_results=3, search_depth='advanced', include_raw_content=True)
web_search_agent = create_react_agent(
    llm, tools=[tavily_tool],
    prompt=prompts.agent_system_prompt('You are the Web Data Extraction Researcher.')
)

def planner_node(state: State) -> Command[Literal["executor"]]:
    plan_instructions = """You are the Mapping Planner.
Break down the user's overarching research query into 2 to 3 distinct, highly targeted sub-queries or URLs.
Return ONLY a JSON list of strings.
User Query: """ + state.get("user_query", "")

    try:
        reply = reasoning_llm.invoke([HumanMessage(content=plan_instructions)])
        content = reply.content.replace('```json', '').replace('```', '')
        tasks = json.loads(content)
        if not isinstance(tasks, list): tasks = [tasks]
    except:
        tasks = [state.get("user_query", "")]
        
    print(f"PLANNER -> Generated {len(tasks)} parallel sub-queries: {tasks}")
    return Command(update={"search_tasks": tasks}, goto="executor")

def executor_node(state: State):
    # Send parallel requests to the web_researcher node for every sub-task
    print("EXECUTOR -> Dispatching parallel Web Researchers...")
    sends = [Send("web_researcher", {"user_query": task}) for task in state.get("search_tasks", [])]
    return Command(goto=sends)

class WebResearcherState(MessagesState):
    user_query: str

def web_research_node(state: WebResearcherState) -> Dict[str, Any]:
    query = state["user_query"]
    print(f"  [Parallel Agent] Searching: {query[:40]}...")
    
    content = ""
    url_match = re.search(r"https?://[^\s]+", query)
    if url_match and "reddit.com" in url_match.group(0):
        try:
            json_url = url_match.group(0).rstrip("/") + ".json?limit=50"
            resp = requests.get(json_url, headers={'User-Agent': 'Mozilla/5.0'})
            if resp.status_code == 200:
                d = resp.json()
                title = d[0]['data']['children'][0]['data'].get('title', '')
                content = f"Reddit Extract: {title}\nURL: {url_match.group(0)}"
        except: pass

    if not content:
        try:
            res = web_search_agent.invoke({"messages": [HumanMessage(content=query)]})
            content = f"Tavily Search for '{query}':\n" + res["messages"][-1].content
        except Exception as e:
            content = f"Search failed for {query}: {e}"
            
    return {"scraped_contexts": [content]} # Using operator.add to aggregate

def synthesizer_node(state: State) -> Command[Literal[END]]:
    print("SYNTHESIZER -> Aggregating all parallel reports...")
    all_contexts = "\n\n======\n\n".join(state.get("scraped_contexts", []))
    prompt = f"User question: {state.get('user_query', '')}\n\nAggregated Context from Parallel Agents:\n\n{all_contexts[:60000]}"
    
    reply = reasoning_llm.invoke([HumanMessage(content=prompt)])
    answer = reply.content
    return Command(update={"final_answer": answer}, goto=END)

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

workflow = StateGraph(State)
workflow.add_node("planner", planner_node)
workflow.add_node("executor", executor_node)
workflow.add_node("web_researcher", web_research_node)
workflow.add_node("synthesizer", synthesizer_node)

workflow.add_edge(START, "planner")
# LangGraph's Send API requires returning sends from a node rather than static edge routing.
# The executor returns multiple `Send` objects aiming at "web_researcher".
workflow.add_edge("web_researcher", "synthesizer")

graph = workflow.compile()

In [ ]:
query = "Research the differences between Llama 3 and Claude 3.5 in terms of coding capabilities. Also look up their pricing models."

state = {
    "user_query": query,
    "search_tasks": [],
    "scraped_contexts": []
}

print("🚀 Starting V5 Pipeline (Parallel Search Mapping)...")
result = graph.invoke(state)

print("\n--- 📝 V5 FINAL AGGREGATED REPORT ---\n")
print(result.get("final_answer"))